# Feed Forward Network with PyTorch

## AI Usage

I used AI to investigate and understand errors. I also used copilot autocomplete here and there for variables names and working with list comprehensions.

## PyTorch setup

In [240]:
import torch

In [241]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)
print(f"Using device: {device}")

Using device: cuda


## Load data and create Datasets

In [242]:
from torchvision.datasets import MNIST
from torchvision import transforms

In [243]:
transform = transforms.ToTensor()
train_data = MNIST(root="data", train=True, download=True, transform=transform)
train_data

Dataset MNIST
    Number of datapoints: 60000
    Root location: data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [244]:
from torch.utils.data import Subset

Split the training data into training and validation sets

In [245]:
train_indices = list(range(50000))
val_indices = list(range(50000, len(train_data)))

val_data = Subset(train_data, val_indices)
train_data = Subset(train_data, train_indices)
print(f"Training set size: {len(train_data)}")
print(f"Validation set size: {len(val_data)}")

Training set size: 50000
Validation set size: 10000


In [246]:
test_data = MNIST(root="data", train=False, download=True, transform=transform)
test_data

Dataset MNIST
    Number of datapoints: 10000
    Root location: data
    Split: Test
    StandardTransform
Transform: ToTensor()

## Defining the Neural Network

In [247]:
import torch.nn as nn

In [248]:
input_size = train_data[0][0].numel()

# I think to(device) is not needed here but I added it just in case
model1 = nn.Sequential(
    nn.Flatten(),
    nn.Linear(input_size, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
).to(device)

model2 = nn.Sequential(
    nn.Flatten(),
    nn.Linear(input_size, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
).to(device)

model3 = nn.Sequential(
    nn.Flatten(),
    nn.Linear(input_size, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10)
).to(device)

## Defining the optimizer

In [249]:
import torch.optim as optim

In [250]:
optimizer1 = optim.Adam(model1.parameters(), lr=0.001)
optimizer2 = optim.Adam(model2.parameters(), lr=0.01)
optimizer3 = optim.SGD(model3.parameters(), lr=0.001)

## Defining loss function

In [251]:
criterion = nn.CrossEntropyLoss()

## Defining mini-batches creation function

In [252]:
import numpy as np

In [253]:
def create_minibatches(batch_size, x, y):
    total_data = len(x)
    indices = np.arange(total_data)

    for i in range(0, total_data, batch_size):
        batch_idx = indices[i:i + batch_size]
        x_batch = torch.stack([x[j] for j in batch_idx])
        y_batch = torch.tensor([y[j] for j in batch_idx], dtype=torch.long)
        yield x_batch, y_batch

## Train loop

In [254]:
from tqdm import tqdm

In [255]:
def train(model, train_data, optimizer, criterion, epochs, batch_size):
    # Training data
    x_data = [x for x, _ in train_data]
    y_data = [y for _, y in train_data]

    # Validation data
    x_val = [x for x, y in val_data]
    y_val = [y for _, y in val_data]

    for epoch in range(epochs):
        # Train
        model.train()
        running_loss = 0.0
        train_correct = 0
        train_total = 0

        epoch_iterator = tqdm(
            create_minibatches(batch_size, x_data, y_data),
            total=(len(train_data) + batch_size - 1) // batch_size,
            desc=f"Epoch {epoch+1}/{epochs}",
            leave=True
        )

        for inputs, labels in epoch_iterator:
            optimizer.zero_grad()
            # The dataset is using cpu, check if we are using gpu and move to gpu
            if inputs.device != device:
                inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            # Accuracy
            preds = outputs.argmax(dim=1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)

            running_loss += loss.item() * len(inputs)

        epoch_train_loss = running_loss / len(train_data)
        epoch_train_accuracy = train_correct / train_total

        # Eval
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in create_minibatches(batch_size, x_val, y_val):
                if inputs.device != device:
                    inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                # Accuracy
                preds = outputs.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

                val_loss += loss.item() * len(inputs)
        
        epoch_val_loss = val_loss / len(val_data)
        epoch_val_accuracy = val_correct / val_total

        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_train_loss:.4f}, Val Loss: {epoch_val_loss:.4f}, Train Acc: {epoch_train_accuracy:.4f}, Val Acc: {epoch_val_accuracy:.4f}")

    return epoch_train_loss, epoch_val_loss, epoch_train_accuracy, epoch_val_accuracy

In [256]:
model1_trained = train(model1, train_data, optimizer1, criterion, epochs=30, batch_size=64)

Epoch 1/30: 100%|██████████| 782/782 [00:02<00:00, 379.95it/s]


Epoch [1/30], Loss: 0.3852, Val Loss: 0.1981, Train Acc: 0.8970, Val Acc: 0.9472


Epoch 2/30: 100%|██████████| 782/782 [00:01<00:00, 447.32it/s]


Epoch [2/30], Loss: 0.1844, Val Loss: 0.1428, Train Acc: 0.9474, Val Acc: 0.9625


Epoch 3/30: 100%|██████████| 782/782 [00:01<00:00, 450.28it/s]


...

Epoch [28/30], Loss: 0.0038, Val Loss: 0.1203, Train Acc: 0.9990, Val Acc: 0.9777


Epoch 29/30: 100%|██████████| 782/782 [00:01<00:00, 464.17it/s]


Epoch [29/30], Loss: 0.0024, Val Loss: 0.1459, Train Acc: 0.9995, Val Acc: 0.9742


Epoch 30/30: 100%|██████████| 782/782 [00:01<00:00, 448.83it/s]


Epoch [30/30], Loss: 0.0047, Val Loss: 0.1424, Train Acc: 0.9987, Val Acc: 0.9741


In [257]:
model2_trained = train(model2, train_data, optimizer2, criterion, epochs=20, batch_size=86)

Epoch 1/20: 100%|██████████| 582/582 [00:01<00:00, 318.62it/s]


Epoch [1/20], Loss: 0.2811, Val Loss: 0.1569, Train Acc: 0.9176, Val Acc: 0.9559


Epoch 2/20: 100%|██████████| 582/582 [00:01<00:00, 347.97it/s]


Epoch [2/20], Loss: 0.1583, Val Loss: 0.1411, Train Acc: 0.9554, Val Acc: 0.9630


Epoch 3/20: 100%|██████████| 582/582 [00:01<00:00, 345.22it/s]


...

Epoch [18/20], Loss: 0.0649, Val Loss: 0.1994, Train Acc: 0.9849, Val Acc: 0.9684


Epoch 19/20: 100%|██████████| 582/582 [00:01<00:00, 351.72it/s]


Epoch [19/20], Loss: 0.0649, Val Loss: 0.2007, Train Acc: 0.9847, Val Acc: 0.9673


Epoch 20/20: 100%|██████████| 582/582 [00:01<00:00, 323.06it/s]


Epoch [20/20], Loss: 0.0573, Val Loss: 0.2177, Train Acc: 0.9869, Val Acc: 0.9680


In [258]:
model3_trained = train(model3, train_data, optimizer3, criterion, epochs=35, batch_size=32)

Epoch 1/35: 100%|██████████| 1563/1563 [00:03<00:00, 489.90it/s]


Epoch [1/35], Loss: 2.2887, Val Loss: 2.2595, Train Acc: 0.1033, Val Acc: 0.1361


Epoch 2/35: 100%|██████████| 1563/1563 [00:03<00:00, 493.33it/s]


Epoch [2/35], Loss: 2.2146, Val Loss: 2.1476, Train Acc: 0.2052, Val Acc: 0.3436


Epoch 3/35: 100%|██████████| 1563/1563 [00:03<00:00, 495.64it/s]


...

Epoch [33/35], Loss: 0.2981, Val Loss: 0.2760, Train Acc: 0.9147, Val Acc: 0.9224


Epoch 34/35: 100%|██████████| 1563/1563 [00:03<00:00, 495.23it/s]


Epoch [34/35], Loss: 0.2948, Val Loss: 0.2731, Train Acc: 0.9157, Val Acc: 0.9225


Epoch 35/35: 100%|██████████| 1563/1563 [00:03<00:00, 487.45it/s]


Epoch [35/35], Loss: 0.2916, Val Loss: 0.2704, Train Acc: 0.9164, Val Acc: 0.9232


In [259]:
for model, (train_loss, val_loss, train_accuracy, val_accuracy) in zip(
    ["Model 1", "Model 2", "Model 3"],
    [model1_trained, model2_trained, model3_trained]
):
    print(f"{model} - Final Training Loss: {train_loss:.4f}, Final Validation Loss: {val_loss:.4f}, Training Accuracy: {train_accuracy:.4f}, Validation Accuracy: {val_accuracy:.4f}")

Model 1 - Final Training Loss: 0.0047, Final Validation Loss: 0.1424, Training Accuracy: 0.9987, Validation Accuracy: 0.9741
Model 2 - Final Training Loss: 0.0573, Final Validation Loss: 0.2177, Training Accuracy: 0.9869, Validation Accuracy: 0.9680
Model 3 - Final Training Loss: 0.2916, Final Validation Loss: 0.2704, Training Accuracy: 0.9164, Validation Accuracy: 0.9232


## Reflection

This activity helped me to have a better understanding of the architecture and flow of a neural network. I learned how to use pythorch datasets and dataloaders and why we have to use dataloaders. I also gained better understanding of basic things like the calculaton of the loss and accuracy for each epoch. in conclusion I liked this activity because I learned how to do things withou libraries and I think I can improve my data processing or training function because I feel like my models were a little slow.